# Stellar Physical Characteristics — Analysis Notebook

## Introduction

This notebook explores the physical properties of stars across different stellar classifications using two publicly available catalogs accessed via the Strasbourg Astronomical Data Center (CDS) TAP service (https://cds.unistra.fr/).

**Data Sources**

| Catalog | CDS ID | Records | Description |
|---------|--------|---------|-------------|
| Hipparcos | I/239/hip_main | ~118,000 | ESA astrometry satellite (1989–1993); sky-wide coverage biased toward bright stars |
| Woolley | V/32A/catalog | ~2,150 | Stars within 25 parsecs; strong coverage of dim, nearby objects |

Using both catalogs is intentional: Hipparcos over-represents luminous stars (they're visible from farther away), while Woolley fills in the dim end of the distribution. Together they give a more complete cross-section.

**Measured Quantities**
- `b_v` — B-V colour index (directly observed; proxy for surface temperature)
- `plx` — trigonometric parallax in milliarcseconds → distance in parsecs via `dist_pc = 1000 / plx`
- `vmag` / `mv` — visual / absolute magnitude

**Stellar Classification**

*Spectral class* (O B A F G K M, hottest to coolest) encodes temperature.  
*Luminosity class* (I–VII) encodes physical size/brightness: I = Supergiants, III = Giants, V = Main Sequence, VII = White Dwarfs.

The Sun is G2V. Polaris A is F7Ib.

---

**Analysis questions answered below**

1. (SQL) Does Hipparcos exhibit luminosity bias compared to the nearby Woolley catalog?
2. (SQL) How does the luminosity class mix change with distance in the combined dataset?
3. (SQL) What are the median colour and absolute magnitude by spectral × luminosity class?
4. (Python/pandas) How does distance-corrected absolute magnitude distribute across luminosity classes?
5. (Python/matplotlib) Hertzsprung-Russell diagrams — full dataset vs per luminosity class


## Setup

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import colorsys
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

DB = "stellar_analysis.db"

def query(sql: str, db: str = DB) -> pd.DataFrame:
    """Execute a SQL query and return results as a DataFrame."""
    with sqlite3.connect(db) as con:
        return pd.read_sql(sql, con)


> **Run ETL first** if the database does not yet exist:
> ```
> python etl.py --db stellar_analysis.db
> ```


In [ ]:
# Confirm tables are present
tables = query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
print("Tables:", tables["name"].tolist())

counts = {
    t: query(f"SELECT COUNT(*) AS n FROM {t}").iloc[0]["n"]
    for t in tables["name"]
}
for t, n in counts.items():
    print(f"  {t}: {n:,} rows")


## Q1 — Does Hipparcos exhibit luminosity bias vs. Woolley? (SQL)

**Rationale:** Hipparcos observed the whole sky but is flux-limited — dimmer stars are
only detectable nearby, so the catalog is skewed toward intrinsically bright objects.
The Woolley catalog is distance-limited (≤25 pc) so it captures dim stars that Hipparcos
misses. Comparing the luminosity-class breakdown as a fraction of each source's total
should make this selection effect visible.

**Expected result:** Hipparcos dominated by Giants (III) and Subgiants (IV); Woolley
dominated by Main Sequence (V) and dim K/M stars.


In [ ]:
SQL_Q1 = """
WITH totals AS (
    SELECT src, COUNT(*) AS total
    FROM stars
    GROUP BY src
),
counts AS (
    SELECT
        src,
        lum_class,
        COUNT(CASE WHEN sp_class = 'O' THEN 1 END) AS O,
        COUNT(CASE WHEN sp_class = 'B' THEN 1 END) AS B,
        COUNT(CASE WHEN sp_class = 'A' THEN 1 END) AS A,
        COUNT(CASE WHEN sp_class = 'F' THEN 1 END) AS F,
        COUNT(CASE WHEN sp_class = 'G' THEN 1 END) AS G,
        COUNT(CASE WHEN sp_class = 'K' THEN 1 END) AS K,
        COUNT(CASE WHEN sp_class = 'M' THEN 1 END) AS M
    FROM stars
    GROUP BY src, lum_class
)
SELECT
    c.src,
    c.lum_class,
    ROUND(100.0 * c.O / t.total, 2) AS O_pct,
    ROUND(100.0 * c.B / t.total, 2) AS B_pct,
    ROUND(100.0 * c.A / t.total, 2) AS A_pct,
    ROUND(100.0 * c.F / t.total, 2) AS F_pct,
    ROUND(100.0 * c.G / t.total, 2) AS G_pct,
    ROUND(100.0 * c.K / t.total, 2) AS K_pct,
    ROUND(100.0 * c.M / t.total, 2) AS M_pct
FROM counts c
JOIN totals t ON c.src = t.src
WHERE lum_class NOT IN ('VI', 'VII', '')
ORDER BY
    c.src,
    CASE c.lum_class
        WHEN 'I'   THEN 1
        WHEN 'II'  THEN 2
        WHEN 'III' THEN 3
        WHEN 'IV'  THEN 4
        WHEN 'V'   THEN 5
        ELSE 6
    END
"""

df_q1 = query(SQL_Q1)
print(df_q1.to_string(index=False))


**Interpretation:** The Hipparcos rows show relatively high percentages of luminosity
classes I–III (giants and supergiants), whereas Woolley is overwhelmingly class V
(main sequence), particularly K and M dwarfs which are the most common star type but
are too faint to feature strongly in a sky-wide magnitude-limited survey.


## Q2 — How does the luminosity-class mix change with distance? (SQL)

**Rationale:** If our combined dataset still has a distance-based selection bias,
we would expect main-sequence stars (V) to dominate nearby distance bins and giants
(III+) to increasingly dominate at larger distances because only very bright objects
are visible at range. Quantifying this slope tells us how quickly the sample becomes
unrepresentative.


In [ ]:
SQL_Q2 = """
WITH totals AS (
    SELECT
        (CAST(dist_pc / 50 AS INT) * 50) AS dist_bucket,
        COUNT(*) AS total
    FROM stars
    WHERE dist_pc IS NOT NULL
    GROUP BY dist_bucket
),
counts AS (
    SELECT
        (CAST(dist_pc / 50 AS INT) * 50) AS dist_bucket,
        COUNT(CASE WHEN lum_class = 'I'   THEN 1 END) AS I,
        COUNT(CASE WHEN lum_class = 'II'  THEN 1 END) AS II,
        COUNT(CASE WHEN lum_class = 'III' THEN 1 END) AS III,
        COUNT(CASE WHEN lum_class = 'IV'  THEN 1 END) AS IV,
        COUNT(CASE WHEN lum_class = 'V'   THEN 1 END) AS V
    FROM stars
    WHERE dist_pc IS NOT NULL
    GROUP BY dist_bucket
)
SELECT
    c.dist_bucket           AS dist_pc_bin,
    t.total,
    ROUND(100.0 * c.I   / t.total, 1) AS I_pct,
    ROUND(100.0 * c.II  / t.total, 1) AS II_pct,
    ROUND(100.0 * c.III / t.total, 1) AS III_pct,
    ROUND(100.0 * c.IV  / t.total, 1) AS IV_pct,
    ROUND(100.0 * c.V   / t.total, 1) AS V_pct
FROM counts c
JOIN totals t ON c.dist_bucket = t.dist_bucket
WHERE c.dist_bucket <= 500
ORDER BY c.dist_bucket
"""

df_q2 = query(SQL_Q2)
print(df_q2.to_string(index=False))


**Interpretation:** Main sequence (V) fraction falls sharply with distance while giant (III)
fraction rises. This mirrors the Malmquist bias: flux-limited surveys preferentially detect
bright objects at large distances. The 0–50 pc bin (dominated by Woolley) is the most
representative snapshot of the true local stellar population.


## Q3 — Median colour and absolute magnitude by spectral × luminosity class (SQL)

**Rationale:** This is the core scientific question. Each combination of spectral class
(temperature proxy) and luminosity class should occupy a distinct region of the
Hertzsprung-Russell diagram. Tabulating medians gives a compact summary and lets us
verify that the data follows known stellar physics (e.g. Giants should be brighter than
Main Sequence stars of the same spectral type).


In [ ]:
SQL_Q3 = """
SELECT
    sp_class,
    lum_class,
    COUNT(*)                        AS n,
    ROUND(AVG(b_v), 3)             AS mean_bv,
    ROUND(AVG(abs_mag), 2)         AS mean_abs_mag,
    ROUND(MIN(abs_mag), 2)         AS brightest,
    ROUND(MAX(abs_mag), 2)         AS dimmest
FROM stars
WHERE lum_class IN ('I', 'II', 'III', 'IV', 'V')
  AND sp_class  IN ('O', 'B', 'A', 'F', 'G', 'K', 'M')
GROUP BY sp_class, lum_class
HAVING COUNT(*) >= 5
ORDER BY
    CASE sp_class
        WHEN 'O' THEN 1 WHEN 'B' THEN 2 WHEN 'A' THEN 3
        WHEN 'F' THEN 4 WHEN 'G' THEN 5 WHEN 'K' THEN 6 WHEN 'M' THEN 7
    END,
    CASE lum_class
        WHEN 'I' THEN 1 WHEN 'II' THEN 2 WHEN 'III' THEN 3
        WHEN 'IV' THEN 4 WHEN 'V' THEN 5
    END
"""

df_q3 = query(SQL_Q3)
print(df_q3.to_string(index=False))


**Interpretation:** Moving down spectral class (O → M), `mean_bv` increases (stars get
redder/cooler) and main-sequence `mean_abs_mag` increases (dimmer). Within a spectral
class, Giants (III) and Supergiants (I) are dramatically brighter (lower `mean_abs_mag`)
than Main Sequence (V) stars — this is the Hertzsprung-Russell diagram in tabular form.


## Q4 — Absolute magnitude distribution by luminosity class (Python/pandas)

**Rationale:** Histograms reveal the shape of the distribution within each class —
whether it is narrow and well-defined (which indicates a physically homogeneous group)
or broad and skewed. This also makes the separation between luminosity classes visually
apparent in magnitude space.


In [ ]:
LUM_ORDER = ["I", "II", "III", "IV", "V"]
LUM_COLORS = {
    "I":   "#ff6b6b",   # Supergiants — red
    "II":  "#ffa94d",   # Bright Giants — orange
    "III": "#ffd43b",   # Giants — yellow
    "IV":  "#74c0fc",   # Subgiants — light blue
    "V":   "#a9e34b",   # Main Sequence — green
}

df_stars = query(
    "SELECT abs_mag, lum_class FROM stars WHERE lum_class IN ('I','II','III','IV','V')"
)

fig, axes = plt.subplots(
    len(LUM_ORDER), 1, figsize=(10, 12), sharex=True, facecolor="#111"
)
fig.patch.set_facecolor("#111")
fig.suptitle("Absolute Magnitude Distribution by Luminosity Class", color="white", fontsize=14)

for ax, lc in zip(axes, LUM_ORDER):
    sub = df_stars[df_stars["lum_class"] == lc]["abs_mag"].dropna()
    ax.hist(sub, bins=60, range=(-12, 20), color=LUM_COLORS[lc], alpha=0.85)
    ax.set_facecolor("#1a1a2e")
    ax.set_ylabel(lc, color="white", rotation=0, labelpad=20, fontsize=12)
    ax.tick_params(colors="white")
    ax.yaxis.set_major_locator(MaxNLocator(3))
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")
    ax.text(
        0.98, 0.75, f"n={len(sub):,}  median={sub.median():.1f}",
        transform=ax.transAxes, ha="right", color="white", fontsize=9
    )

axes[-1].set_xlabel("Absolute Magnitude (brighter ←  → dimmer)", color="white")
plt.tight_layout()
plt.savefig("magnitude_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


**Interpretation:**
- Supergiants (I) cluster around Mv ≈ –5 to –8 — extremely luminous outliers.
- Giants (III) form a broad distribution from ~0 to –3.
- Main sequence (V) spans the widest range (~2 to 16), reflecting the enormous temperature
  spread from hot O/B dwarfs to cool M dwarfs.
- The cleaner the peak, the more physically uniform that class is.


## Q5 — Hertzsprung-Russell diagrams by luminosity class (Python/matplotlib)

**Rationale:** The HR diagram (colour vs. absolute magnitude) is the central tool of
stellar astrophysics. Plotting each luminosity class separately on a distance-limited
subset makes the loci of each class visible without the contamination that occurs when
all classes are overplotted. Point colour is derived from B-V via Ballesteros' formula
to approximate each star's true visual colour.


In [ ]:
def bv_to_rgb(bv: float, saturation_boost: float = 1.5) -> tuple[float, float, float]:
    """
    Convert B-V colour index to an approximate RGB colour.

    Steps:
    1. Ballesteros' formula: B-V -> temperature (K)
       Source: https://en.wikipedia.org/wiki/Color_index
    2. Temperature -> RGB (Tanner Helland algorithm)
       Source: https://tannerhelland.com/2012/09/18/convert-temperature-rgb-algorithm-code.html
    3. Saturation boost via HSV roundtrip to make colours more vivid.
    """
    bv = np.clip(bv, -0.4, 2.0)
    t = 4600 * (1 / (0.92 * bv + 1.7) + 1 / (0.92 * bv + 0.62))
    t = np.clip(t, 1000, 40000) / 100.0

    r = 255 if t <= 66 else 329.699 * ((t - 60) ** -0.133)
    g = 99.471 * np.log(t) - 161.120 if t <= 66 else 288.122 * ((t - 60) ** -0.0756)
    b = 0 if t <= 19 else (255 if t >= 66 else 138.518 * np.log(t - 10) - 305.0448)

    r, g, b = (np.clip(x, 0, 255) / 255 for x in (r, g, b))
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    return colorsys.hsv_to_rgb(h, min(1.0, s * saturation_boost), v)


In [ ]:
# Distance limits per luminosity class keep each panel from becoming sparse or dominated by noise.
# Giants and supergiants are visible from much farther out than dwarfs.
CONFIGS = [
    ("All",  None,  None,  "All Stars (combined)"),
    ("I",    None,  1000,  "Supergiants (I)"),
    ("II",   None,  500,   "Bright Giants (II)"),
    ("III",  None,  100,   "Giants (III)"),
    ("IV",   None,  200,   "Subgiants (IV)"),
    ("V",    None,  50,    "Main Sequence (V)"),
]

datasets = []
for lc, min_d, max_d, label in CONFIGS:
    clauses = []
    if lc != "All":
        clauses.append(f"lum_class = '{lc}'")
    if min_d is not None:
        clauses.append(f"dist_pc >= {min_d}")
    if max_d is not None:
        clauses.append(f"dist_pc <= {max_d}")
    where = ("WHERE " + " AND ".join(clauses)) if clauses else ""
    df = query(f"SELECT b_v, abs_mag FROM stars {where}")
    datasets.append((df, label))
    print(f"{label}: {len(df):,} stars")


In [ ]:
def plot_hr_panels(datasets, ncols: int = 2) -> None:
    """Render a grid of HR diagram panels, one per dataset."""
    nrows = -(-len(datasets) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 5), facecolor="black")
    fig.patch.set_facecolor("black")

    for ax, (df_sub, title) in zip(axes.flat, datasets):
        valid = df_sub.dropna(subset=["b_v", "abs_mag"])
        colors = valid["b_v"].apply(bv_to_rgb).tolist()
        ax.set_facecolor("black")
        ax.scatter(valid["b_v"], valid["abs_mag"],
                   c=colors, alpha=0.3, s=1, marker=",")
        ax.set_title(title, color="white", fontsize=10)
        ax.set_xlabel("B-V  (blue ← → red)", color="white", fontsize=8)
        ax.set_ylabel("Absolute Magnitude", color="white", fontsize=8)
        ax.tick_params(colors="white", labelsize=7)
        ax.set_xlim(-0.5, 2.1)
        ax.set_ylim(18, -12)
        ax.text(0.98, 0.02, f"n={len(valid):,}", transform=ax.transAxes,
                ha="right", color="white", fontsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")

    for ax in axes.flat[len(datasets):]:
        ax.set_visible(False)

    plt.suptitle("Hertzsprung-Russell Diagrams", color="white", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig("hr_diagrams.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_hr_panels(datasets, ncols=2)


**Interpretation:**
- **All Stars** panel shows the classic HR diagram: a diagonal main sequence (bottom-left
  to upper-right), a giant branch (upper-right clump), and faint white dwarfs (bottom).
- **Main Sequence (V)** restricted to 50 pc shows a clean diagonal from hot blue A stars
  to cool red M dwarfs, with the Sun-like G dwarfs in the middle.
- **Giants (III)** cluster tightly at moderate B-V and Mv ≈ 0 to –3, confirming their
  physical homogeneity compared to the main sequence.
- **Supergiants (I)** span a wide B-V range at very negative absolute magnitudes — the
  rarest and most extreme objects in the dataset.
